# 02 · HOG + linear SVM baseline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/02_hog_svm.ipynb)

Классический baseline. Обучение только на оригинальных train-изображениях, `C` выбирается по validation.
Тест здесь **не** запускается — финальная оценка всех моделей в `04_evaluate`.

GPU не нужен (можно CPU runtime). Модель сохраняется в `MyDrive/bgbias/runs/hog_svm/`.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
REPO_URL = "https://github.com/mundmina/dl_project.git"
REPO_DIR = "/content/dl_project"
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:  # always run the latest code from GitHub
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    from google.colab import userdata
    userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import json, time
import joblib
import numpy as np
import pandas as pd
from bgbias import hog_svm
from bgbias.project import make_dataset

RUN_DIR = RUNS / "hog_svm"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE = Path("/content/cache/hog")   # features are large, keep them on local disk only
CACHE.mkdir(parents=True, exist_ok=True)

def features(split, condition):
    path = CACHE / f"{split}_{condition}.npy"
    if path.exists():
        return np.load(path)
    t0 = time.time()
    X = hog_svm.extract(make_dataset(data, split, condition=condition, transform=None))
    np.save(path, X)
    print(f"{split}/{condition}: {X.shape} in {time.time() - t0:.0f}s")
    return X

X_train, y_train = features("train", "original"), data.splits["train"].label.to_numpy()
X_val, y_val = features("val", "original"), data.splits["val"].label.to_numpy()

## Подбор C по validation

In [ ]:
rows = []
for C in hog_svm.C_GRID:
    t0 = time.time()
    clf = hog_svm.make_svm(C).fit(X_train, y_train)
    rows.append({"C": C, "train_acc": clf.score(X_train, y_train), "val_acc": clf.score(X_val, y_val),
                 "fit_s": round(time.time() - t0)})
    print(rows[-1])
grid = pd.DataFrame(rows)
best_C = float(grid.loc[grid.val_acc.idxmax(), "C"])
grid.to_csv(RUN_DIR / "c_grid.csv", index=False)
print("best C:", best_C)
grid

## Финальная модель + проверка на val swapped (только для информации)

In [ ]:
clf = hog_svm.make_svm(best_C).fit(X_train, y_train)
X_val_swapped = features("val", "swapped")
val_metrics = {
    "C": best_C,
    "val_acc_original": clf.score(X_val, y_val),
    "val_acc_swapped": clf.score(X_val_swapped, y_val),
}
joblib.dump(clf, RUN_DIR / "model.joblib")
(RUN_DIR / "val_metrics.json").write_text(json.dumps(val_metrics, indent=1))
val_metrics